Network modularity test. 

In [1]:
# one subjects time series from the data



In [ ]:
!pip install bctpy netneurotools scipy statsmodels pandas numpy

In [4]:
# network modularity test code 
import numpy as np
from netneurotools.modularity import consensus_modularity


def signed_asymmetric_q(W, communities, gamma=1.0):
    """
    Evaluate signed asymmetric modularity Q for a fixed partition.

    Matches the negative_asym objective implemented by
    bctpy.community_louvain.
    """
    W = np.asarray(W, dtype=float).copy()
    ci = np.asarray(communities).reshape(-1)

    if W.ndim != 2 or W.shape[0] != W.shape[1]:
        raise ValueError("FC matrix must be square.")

    if len(ci) != W.shape[0]:
        raise ValueError("One community label is required per ROI.")

    if not np.all(np.isfinite(W)):
        raise ValueError("FC matrix contains NaN or Inf.")

    # Ensure an undirected matrix
    W = (W + W.T) / 2
    np.fill_diagonal(W, 0)

    # Positive and absolute negative weights
    W_pos = np.maximum(W, 0)
    W_neg = np.maximum(-W, 0)

    s_pos = W_pos.sum()
    s_neg = W_neg.sum()

    if s_pos == 0:
        raise ValueError("No positive weights in FC matrix.")

    k_pos = W_pos.sum(axis=1)

    B_pos = (
        W_pos
        - gamma * np.outer(k_pos, k_pos) / s_pos
    )

    if s_neg > 0:
        k_neg = W_neg.sum(axis=1)

        B_neg = (
            W_neg
            - gamma * np.outer(k_neg, k_neg) / s_neg
        )
    else:
        B_neg = np.zeros_like(W)

    # Rubinov-Sporns asymmetric signed objective
    B_signed = (
        B_pos / s_pos
        - B_neg / (s_pos + s_neg)
    )

    same_module = ci[:, None] == ci[None, :]

    return float(B_signed[same_module].sum())


def subject_consensus_modularity(
    roi_timeseries,
    repeats=100,
    gamma=1.0,
    seed=12345,
):
    """
    Parameters
    ----------
    roi_timeseries : array, shape (timepoints, ROIs)

    Returns
    -------
    consensus_q
        Modularity Q of the final consensus partition.
    consensus_ci
        Final community assignment for each ROI.
    q_runs
        Q from each of the 100 Louvain optimizations.
    """
    ts = np.asarray(roi_timeseries, dtype=float)

    if ts.ndim != 2:
        raise ValueError(
            "Time series must have shape (timepoints, ROIs)."
        )

    # Pearson correlation between ROI time series
    fc = np.corrcoef(ts, rowvar=False)

    if not np.all(np.isfinite(fc)):
        raise ValueError(
            "FC contains NaN/Inf. Check for constant ROI time series."
        )

    fc = (fc + fc.T) / 2
    np.fill_diagonal(fc, 0)

    # 100 signed Louvain runs followed by consensus clustering
    consensus_ci, q_runs, zrand = consensus_modularity(
        adjacency=fc,
        gamma=gamma,
        B="negative_asym",
        repeats=repeats,
        seed=seed,
    )

    # Q of the resultant consensus partition
    consensus_q = signed_asymmetric_q(
        fc,
        consensus_ci,
        gamma=gamma,
    )

    return {
        "Q": consensus_q,
        "communities": consensus_ci,
        "FC": fc,
        "Q_runs": q_runs,
        "mean_optimization_Q": float(np.mean(q_runs)),
        "sd_optimization_Q": float(np.std(q_runs, ddof=1)),
        "zrand": zrand,
    }

In [ ]:
ts_path = ""
roi_ts = np.load(ts_path)
# Expected shape: timepoints × ROIs

result = subject_consensus_modularity(
    roi_timeseries=roi_ts,
    repeats=100,
    gamma=1.0,
    seed=12345,
)

print("Final consensus Q:", result["Q"])
print("Number of communities:",
      np.unique(result["communities"]).size)

print("Mean Q from optimization runs:",
      result["mean_optimization_Q"])

In [ ]:
print(f"Mean Q from optimization runs: {result['mean_optimization_Q']}")
print(f"end=> Final consensus Q: {result['Q']}")